<a href="https://colab.research.google.com/github/pksheaad/Transformers/blob/main/02_Dataset_Dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
# importing libraries
import requests
import torch
import torch.nn as nn
from torch.utils.data.dataset import Dataset
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import RandomSampler

In [3]:
# retrive the data
url = "https://raw.githubusercontent.com/pksheaad/Transformers/refs/heads/main/Data/tiny-shakespeare.txt"

request = requests.get(url = url)
text = request.text
print(len(text))

1115394


In [48]:
# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [6]:
# Creating encoder and decoder
class CharTokenizer:
  def __init__(self, vocabulary) -> None:
    self.token_id_for_char = {char : token_id for token_id, char in enumerate(vocabulary)}
    self.cah_for_token_id = {token_id : char for token_id, char in enumerate(vocabulary)}

  @staticmethod
  def trian_to_text(text):
    vocabulary = set(text)

    return CharTokenizer(sorted(list(vocabulary)))

  def encode(self,text):
    """ This method will take the text and return the tensor of token_ids"""
    token_ids = []
    for char in text:
      token = self.token_id_for_char[char]
      token_ids.append(token)

    return torch.tensor(token_ids,dtype = torch.long)

  def decode(self, token_ids):
    decode_text = []
    for token in token_ids.tolist():
      char = self.cah_for_token_id[token]
      decode_text.append(char)
    return "".join(decode_text)

  def get_length(self):
    return len(self.token_id_for_char)

In [32]:
# Testing
tokenize = CharTokenizer.trian_to_text(text = text)
print(tokenize.encode(text = text)[:10])
#print(tokenize.decode(tokenize.encode(text = text)))
print(tokenize.get_length())

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])
65


#Create the Dataset class
from torch.utils.data import Dataset

In this class we basically implement three method of Datadet class


1.   **__init__()** constructor This will use for datastore and transform

1.   **__len__()** method Return Number of element in dataset
2.   **__getitem__()** Return an item from the provided position, passed as index.


2.   List item



In [54]:
class TokenIdsDataset(Dataset):
  def __init__(self, data, block_size: int):
    super().__init__()
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, index):
    assert index < len(self.data) -self.block_size

    x = self.data[index: index + self.block_size]
    y = self.data[index + 1: index + 1 + self.block_size]
    return x, y



In [55]:
# testing
tokenizer = CharTokenizer.trian_to_text(text = text)
encoder = tokenize.encode(text = text)
dataset = TokenIdsDataset(data = encoder, block_size = 64)
print(dataset[0])
x, y = dataset[0]
print(tokenize.decode(x))
print(tokenizer.decode(y))

(tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50]), tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53,
        56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,  1,
        44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1, 57,
        54, 43, 39, 49,  8,  0,  0, 13, 50, 50]))
First Citizen:
Before we proceed any further, hear me speak.

Al
irst Citizen:
Before we proceed any further, hear me speak.

All


In [44]:
#Create RandomSampler
sampler = RandomSampler(data_source = dataset, replacement = True ) # Random sampleing of the data and Replacement = True mean same data can be select multiple times

In [45]:
dataloader = DataLoader(dataset = dataset,
                        batch_size = 64,
                        sampler = sampler)


In [46]:
x, y = next(iter(dataloader))
print(x[0], y[0])
print(x.shape, y.shape)

tensor([43, 52,  1, 61, 53, 59, 50, 42,  6,  0, 13, 52, 42,  1, 61, 43]) tensor([52,  1, 61, 53, 59, 50, 42,  6,  0, 13, 52, 42,  1, 61, 43,  1])
torch.Size([64, 16]) torch.Size([64, 16])
